# Chapter 12.5 — Lab: Unsupervised Learning
### *An Introduction to Statistical Learning* (ISLP, Python edition)

This notebook works through all four parts of the lab:

1. **12.5.1** — Principal Components Analysis (`USArrests`)
2. **12.5.2** — Matrix Completion (using PCA to impute missing values)
3. **12.5.3** — Clustering (K-Means & Hierarchical, on simulated data)
4. **12.5.4** — NCI60 Example (PCA + clustering on real gene-expression data)

Each section has a short explanation before the code, and comments inside the code cells explaining what each line does.


## Setup

Run this first. It imports every library used anywhere in the notebook.

> If `ISLP` is not installed, uncomment the pip install line below.


In [ ]:
# !pip install ISLP

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.datasets import get_rdataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, cut_tree

from ISLP import load_data
from ISLP.cluster import compute_linkage

# We'll use this shorthand name for hierarchical clustering, matching the book
HClust = AgglomerativeClustering

%matplotlib inline


---
## 12.5.1 Principal Components Analysis

**Dataset:** `USArrests` — 50 U.S. states (rows), 4 variables (columns): `Murder`, `Assault`, `UrbanPop`, `Rape`.

**Goal:** Reduce 4 correlated variables down to a couple of "summary axes" (principal components) that capture most of the variation in the data, so we can visualize and interpret the states more easily.


In [ ]:
USArrests = get_rdataset('USArrests').data
USArrests

### Step 1 — Look at the raw scales

The variables have very different means and variances (e.g. `Assault` is measured in the hundreds while `Murder` is in single digits). If we don't standardize, PCA will be dominated by whichever variable happens to have the largest numeric scale — not necessarily the most informative one.


In [ ]:
print("Column means:\n", USArrests.mean(), sep="")
print("\nColumn variances:\n", USArrests.var(), sep="")

### Step 2 — Standardize, then run PCA

Standardizing sets every column to mean 0, standard deviation 1, so all four variables are on equal footing before PCA looks for directions of maximum variance.


In [ ]:
scaler = StandardScaler(with_mean=True, with_std=True)
USArrests_scaled = scaler.fit_transform(USArrests)

pcaUS = PCA()
scores = pcaUS.fit_transform(USArrests_scaled)   # the principal component scores (new coordinates)

# The "loadings": how much each original variable contributes to each PC
loadings = pd.DataFrame(
    pcaUS.components_.T,
    index=USArrests.columns,
    columns=[f'PC{i+1}' for i in range(4)]
)
loadings

### Step 3 — Biplot: visualize states + variable loadings together

A biplot overlays the PC1/PC2 scores (states) with arrows showing how the original variables load onto those same two axes.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(scores[:, 0], scores[:, 1], alpha=0.0)  # invisible points, just to set axis scale

for i, state in enumerate(USArrests.index):
    ax.text(scores[i, 0], scores[i, 1], state, ha='center', va='center', fontsize=8, color='steelblue')

scale_arrow = 7  # purely visual scaling factor for the loading arrows
for j, var in enumerate(USArrests.columns):
    ax.arrow(0, 0, loadings.iloc[j, 0] * scale_arrow, loadings.iloc[j, 1] * scale_arrow,
              color='firebrick', head_width=0.15, alpha=0.8)
    ax.text(loadings.iloc[j, 0] * scale_arrow * 1.15, loadings.iloc[j, 1] * scale_arrow * 1.15,
            var, color='firebrick', fontsize=11, ha='center')

ax.axhline(0, color='grey', lw=0.5)
ax.axvline(0, color='grey', lw=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Biplot: USArrests, PC1 vs PC2')
plt.show()

### Step 4 — How much variance does each component explain?

This tells us how much information we'd lose by keeping only the first few components instead of all four original variables.


In [ ]:
print("Explained variance (raw):", pcaUS.explained_variance_)
print("Explained variance ratio: ", pcaUS.explained_variance_ratio_)
print("Cumulative variance ratio:", np.cumsum(pcaUS.explained_variance_ratio_))

In [ ]:
ticks = np.arange(pcaUS.n_components_) + 1
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ticks, pcaUS.explained_variance_ratio_, marker='o')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Proportion of Variance Explained')
axes[0].set_ylim(0, 1)
axes[0].set_xticks(ticks)
axes[0].set_title('Scree Plot')

axes[1].plot(ticks, np.cumsum(pcaUS.explained_variance_ratio_), marker='o', color='darkorange')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Proportion of Variance Explained')
axes[1].set_ylim(0, 1)
axes[1].set_xticks(ticks)
axes[1].set_title('Cumulative Variance Explained')

plt.tight_layout()
plt.show()

**Reading the plot:** PC1 alone explains ~62% of the total variation, and PC1+PC2 together explain ~87%. So a 2-dimensional picture (the biplot above) is already a very faithful summary of the original 4-dimensional data.


---
## 12.5.2 Matrix Completion

**Idea:** Real datasets often have missing values. Instead of just filling gaps with the column average, we can exploit *correlations between variables* to make smarter guesses — using PCA. This is the essence of **matrix completion**.

**Algorithm 12.1 (in words):**
1. Fill missing entries with column means as a starting guess.
2. Reconstruct the whole matrix using only the top `M` principal components (this "denoises" and re-estimates every entry, including the missing ones).
3. Re-insert those reconstructed values into just the missing spots.
4. Repeat steps 2–3 until the reconstruction barely changes anymore.

We'll test this by **artificially deleting** some known values from `USArrests`, running the algorithm, and then checking how close the recovered values are to the true ones.


In [ ]:
X = USArrests_scaled.copy()

# Randomly hide 20 of the 200 entries (10%), spreading them across different rows
n_omit = 20
np.random.seed(15)
r_idx = np.random.choice(np.arange(X.shape[0]), n_omit, replace=False)  # which states
c_idx = np.random.choice(np.arange(X.shape[1]), n_omit, replace=True)   # which variable in each state

Xna = X.copy()
Xna[r_idx, c_idx] = np.nan
ismiss = np.isnan(Xna)

print(f"Number of missing entries: {ismiss.sum()}")

In [ ]:
def low_rank(X, M=1):
    """Reconstruct X using only the top M singular components (i.e. top M principal components)."""
    U, D, V = np.linalg.svd(X)
    L = U[:, :M] * D[None, :M]
    return L.dot(V[:M])

# Step 1: initialize missing entries with column means (computed from the non-missing values)
Xhat = Xna.copy()
Xbar = np.nanmean(Xhat, axis=0)
Xhat[ismiss] = np.take(Xbar, c_idx)

thresh = 1e-7
rel_err = 1
count = 0
mss0 = np.mean(Xna[~ismiss] ** 2)     # scale factor, used to normalize the relative error
mssold = np.mean(Xhat[~ismiss] ** 2)

# Step 2: iterate -> reconstruct with M=1 PC, refill missing spots, check convergence
while rel_err > thresh:
    count += 1
    Xapp = low_rank(Xhat, M=1)
    Xhat[ismiss] = Xapp[ismiss]
    mss = np.mean(((Xna - Xapp)[~ismiss]) ** 2)
    rel_err = (mssold - mss) / mss0
    mssold = mss
    print(f"Iteration {count}: MSS = {mss:.5f}, Rel. Err = {rel_err:.2e}")

print("\nConverged.")

### Check the recovery quality

Since we know the true values (we deleted them ourselves), we can directly compare true vs. imputed values.


In [ ]:
corr = np.corrcoef(Xapp[ismiss], X[ismiss])[0, 1]
print(f"Correlation between true and imputed values: {corr:.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X[ismiss], Xapp[ismiss], alpha=0.7)
lims = [min(X[ismiss].min(), Xapp[ismiss].min()), max(X[ismiss].max(), Xapp[ismiss].max())]
ax.plot(lims, lims, 'r--', lw=1)  # y = x reference line
ax.set_xlabel('True (deleted) value')
ax.set_ylabel('Imputed value')
ax.set_title(f'Matrix Completion Recovery (corr = {corr:.2f})')
plt.show()

A correlation around 0.6–0.7 between the true and imputed values shows the PCA-based approach recovers real signal, doing meaningfully better than naive mean-filling would.


---
## 12.5.3 Clustering

Two different strategies for grouping observations, both fully unsupervised.

### K-Means Clustering

You choose the number of clusters `K` up front. The algorithm assigns every point to whichever of `K` cluster centers minimizes within-cluster variance.

We'll first build a small simulated dataset where we *know* there are really 2 clusters (by construction), to see the method work as expected.


In [ ]:
np.random.seed(0)
X_sim = np.random.standard_normal((50, 2))
X_sim[:25, 0] += 3   # shift the first 25 points to create a genuine second cluster
X_sim[:25, 1] -= 4

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_sim[:, 0], X_sim[:, 1])
ax.set_title('Simulated data (true structure: 2 groups)')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=2, n_init=20).fit(X_sim)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_sim[:, 0], X_sim[:, 1], c=kmeans.labels_, cmap='viridis')
ax.set_title('K-Means Clustering Results, K=2')
plt.show()

print("Cluster labels:", kmeans.labels_)
print("Total within-cluster sum of squares (inertia):", kmeans.inertia_)

**Why `n_init=20`?** K-means starts from *random* initial cluster centers and can converge to a poor local solution depending on that random start. `n_init` reruns the whole algorithm from multiple random starts and keeps the best (lowest inertia) result. Always use a reasonably large value (20–50) — never rely on a single run.

Let's confirm this by comparing `n_init=1` vs `n_init=20`:


In [ ]:
km1 = KMeans(n_clusters=3, random_state=3, n_init=1).fit(X_sim)
km20 = KMeans(n_clusters=3, random_state=3, n_init=20).fit(X_sim)

print(f"Inertia with n_init=1:  {km1.inertia_:.2f}")
print(f"Inertia with n_init=20: {km20.inertia_:.2f}")
print("(Lower inertia = better clustering; n_init=20 should be at least as good.)")

### Hierarchical Clustering

Instead of fixing `K` up front, we build a full tree ("dendrogram") of nested clusters, then decide afterward how many clusters we want by cutting the tree at some height.

Key choice: **linkage** — how to measure distance *between* clusters (not just points):
- **Complete linkage** — farthest pair of points between the two clusters
- **Single linkage** — closest pair of points (tends to produce long, chained clusters)
- **Average linkage** — average distance across all cross-cluster pairs


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cargs = {'color_threshold': -np.inf, 'above_threshold_color': 'black'}

for ax, method in zip(axes, ['complete', 'average', 'single']):
    hc = HClust(distance_threshold=0, n_clusters=None, linkage=method).fit(X_sim)
    linkage_matrix = compute_linkage(hc)
    dendrogram(linkage_matrix, ax=ax, **cargs)
    ax.set_title(f'{method.capitalize()} Linkage')

plt.tight_layout()
plt.show()

### Cutting the dendrogram into clusters

`cut_tree` lets us pick a specific number of clusters after the fact.


In [ ]:
hc_complete = HClust(distance_threshold=0, n_clusters=None, linkage='complete').fit(X_sim)
linkage_complete = compute_linkage(hc_complete)

for k in [2, 3, 4]:
    labels = cut_tree(linkage_complete, n_clusters=k).reshape(-1)
    print(f"K={k} cluster sizes:", pd.Series(labels).value_counts().to_dict())

**Key takeaway:** K-means and hierarchical clustering, even asked for the same number of clusters on the same data, will often disagree with each other to some extent. Neither is automatically "the truth" — clustering is an exploratory tool, and results should be treated as a starting hypothesis, not a final answer.


---
## 12.5.4 NCI60 Data Example

**Dataset:** 64 cancer cell lines, each with 6,830 gene expression measurements. Each cell line also has a *known* cancer-type label (breast, leukemia, melanoma, etc.) — but we **do not use these labels during clustering**. We only check afterward whether our unsupervised results happen to line up with the known biology, as a sanity check.

This is a realistic example of combining PCA + clustering on high-dimensional data.


In [ ]:
NCI60 = load_data('NCI60')
nci_labs = NCI60['labels']
nci_data = NCI60['data']

print("Data shape (cell lines x genes):", nci_data.shape)
nci_labs.value_counts()

### PCA for visualization

With 6,830 variables we obviously can't plot the raw data. We scale it and run PCA, then plot the first couple of principal component scores — colored by the *known* cancer type purely to see whether similar cancers cluster together.


In [ ]:
scaler_nci = StandardScaler()
nci_scaled = scaler_nci.fit_transform(nci_data)

nci_pca = PCA()
nci_scores = nci_pca.fit_transform(nci_scaled)

# Assign a distinct color to each cancer type for the plot
cancer_types = nci_labs['label'].unique()
color_map = {ct: plt.cm.tab20(i / len(cancer_types)) for i, ct in enumerate(cancer_types)}
colors = nci_labs['label'].map(color_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(nci_scores[:, 0], nci_scores[:, 1], c=colors)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('NCI60: PC1 vs PC2')

axes[1].scatter(nci_scores[:, 0], nci_scores[:, 2], c=colors)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC3')
axes[1].set_title('NCI60: PC1 vs PC3')

plt.tight_layout()
plt.show()

In [ ]:
# How much variance do the first several components explain?
print("Cumulative variance explained by first 10 PCs:")
print(np.cumsum(nci_pca.explained_variance_ratio_)[:10])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(np.arange(1, 11), nci_pca.explained_variance_ratio_[:10], marker='o')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Proportion of Variance Explained')
ax.set_title('NCI60 Scree Plot (first 10 PCs)')
plt.show()

### Hierarchical clustering on the full gene-expression data

We cluster the 64 cell lines using complete linkage on all 6,830 genes, cut the tree into 4 clusters, and cross-tabulate against the *known* cancer types to check for agreement.


In [ ]:
hc_nci = HClust(linkage='complete', distance_threshold=0, n_clusters=None).fit(nci_scaled)
linkage_nci = compute_linkage(hc_nci)

fig, ax = plt.subplots(figsize=(10, 8))
dendrogram(linkage_nci, labels=np.asarray(nci_labs['label']), leaf_font_size=8, ax=ax, **cargs)
ax.set_title('Hierarchical Clustering (Complete Linkage) on NCI60')
plt.tight_layout()
plt.show()

In [ ]:
comp_cut = cut_tree(linkage_nci, n_clusters=4).reshape(-1)
pd.crosstab(nci_labs['label'], pd.Series(comp_cut, name='HClust Cluster'))

**Reading this table:** notice how all the leukemia cell lines fall into a single cluster, while breast cancer cell lines are spread across several — a genuine biological finding (breast cancer is known to be genetically heterogeneous), discovered without ever telling the algorithm what cancer type anything was.

### Compare against K-means (K=4) on the same data


In [ ]:
nci_kmeans = KMeans(n_clusters=4, random_state=0, n_init=20).fit(nci_scaled)

pd.crosstab(
    pd.Series(comp_cut, name='HClust'),
    pd.Series(nci_kmeans.labels_, name='K-means')
)

As before, the two methods agree on some clusters and disagree on others — expected behavior, not an error.

### Clustering on PCA scores instead of raw data (denoising)

With 6,830 noisy gene measurements, clustering on just the first 5 principal components can work *better* than clustering on the raw data, because PCA has already filtered out noise and kept the dominant signal.


In [ ]:
hc_pca = HClust(n_clusters=None, distance_threshold=0, linkage='complete').fit(nci_scores[:, :5])
linkage_pca = compute_linkage(hc_pca)

fig, ax = plt.subplots(figsize=(10, 8))
dendrogram(linkage_pca, labels=np.asarray(nci_labs['label']), leaf_font_size=8, ax=ax, **cargs)
ax.set_title('Hierarchical Clustering on First 5 Principal Component Scores')
plt.tight_layout()
plt.show()

pca_cut = cut_tree(linkage_pca, n_clusters=4).reshape(-1)
pd.crosstab(nci_labs['label'], pd.Series(pca_cut, name='Complete-PCA Cluster'))

**Takeaway:** clustering on principal components rather than the full raw feature set can be viewed as a denoising step — a useful trick any time you have many noisy, correlated features (as is common in genomics).

---
## Summary

| Section | Technique | Key idea |
|---|---|---|
| 12.5.1 | PCA | Compress correlated variables into a few informative axes |
| 12.5.2 | Matrix Completion | Use PCA's low-rank structure to impute missing values smartly |
| 12.5.3 | K-Means / Hierarchical | Two different ways to discover unlabeled groups in data |
| 12.5.4 | NCI60 | Combine PCA + clustering on high-dimensional real data, and validate against (unused) known labels |

This closes out the ISLP Chapter 12.5 lab.
